# VLA 训练教程

本教程将带你了解如何训练 VLA 模型，包括：
- 训练流程
- 损失函数
- 优化器配置
- 训练技巧

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import sys
sys.path.insert(0, '/root/.openclaw/workspace/vla-training')

print("✅ 环境准备完成！")

## 1. 训练流程概述

```
1. 加载数据
2. 前向传播 → 计算损失
3. 反向传播 → 计算梯度
4. 优化器更新参数
5. 记录日志
6. 保存检查点
```

## 2. 损失函数

In [ ]:
# Flow Matching 损失
class FlowMatchingLoss(nn.Module):
    """
    流匹配损失
    
    目标：学习从噪声到动作的速度场
    """
    def __init__(self):
        super().__init__()
    
    def forward(self, pred_velocity, target_velocity):
        """
        Args:
            pred_velocity: 预测速度 (B, T, D)
            target_velocity: 目标速度 (B, T, D)
        """
        return torch.mean((pred_velocity - target_velocity) ** 2)


# Diffusion 损失
class DiffusionLoss(nn.Module):
    """
    扩散损失
    
    目标：预测添加的噪声
    """
    def __init__(self):
        super().__init__()
    
    def forward(self, pred_noise, target_noise):
        return torch.mean((pred_noise - target_noise) ** 2)


# MSE 损失（用于 MLP 动作头）
class MSELoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.mse = nn.MSELoss()
    
    def forward(self, pred_action, target_action):
        return self.mse(pred_action, target_action)


# 测试损失函数
batch_size, chunk_size, action_dim = 4, 10, 7

pred = torch.randn(batch_size, chunk_size, action_dim)
target = torch.randn(batch_size, chunk_size, action_dim)

fm_loss = FlowMatchingLoss()(pred, target)
diff_loss = DiffusionLoss()(pred, target)
mse_loss = MSELoss()(pred, target)

print(f"Flow Matching Loss: {fm_loss.item():.4f}")
print(f"Diffusion Loss: {diff_loss.item():.4f}")
print(f"MSE Loss: {mse_loss.item():.4f}")

## 3. 优化器配置

In [ ]:
# 创建模拟模型
class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer = nn.Linear(100, 70)
    
    def forward(self, x):
        return self.layer(x)

model = SimpleModel()

# AdamW 优化器（推荐）
optimizer = optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=0.1,
    betas=(0.9, 0.95)
)

# 学习率调度器
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=100,
    eta_min=1e-5
)

print("✅ 优化器配置完成！")
print(f"初始学习率: {optimizer.param_groups[0]['lr']:.6f}")

## 4. 训练循环

In [ ]:
def train_step(model, batch, optimizer, criterion, device='cpu'):
    """
    单步训练
    """
    model.train()
    
    # 获取数据
    images = batch['images'].to(device)
    actions = batch['actions'].to(device)
    
    # 前向传播
    predictions = model(images)
    
    # 计算损失
    loss = criterion(predictions, actions)
    
    # 反向传播
    optimizer.zero_grad()
    loss.backward()
    
    # 梯度裁剪
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    
    # 更新参数
    optimizer.step()
    
    return loss.item()


def validate(model, dataloader, criterion, device='cpu'):
    """
    验证
    """
    model.eval()
    total_loss = 0
    
    with torch.no_grad():
        for batch in dataloader:
            images = batch['images'].to(device)
            actions = batch['actions'].to(device)
            
            predictions = model(images)
            loss = criterion(predictions, actions)
            total_loss += loss.item()
    
    return total_loss / len(dataloader)

print("✅ 训练和验证函数定义完成！")

## 5. 模拟训练

In [ ]:
# 模拟训练过程
def simulate_training(num_epochs=10, steps_per_epoch=20):
    """模拟训练过程"""
    
    train_losses = []
    val_losses = []
    learning_rates = []
    
    model = SimpleModel()
    optimizer = optim.AdamW(model.parameters(), lr=3e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    criterion = MSELoss()
    
    for epoch in range(num_epochs):
        # 训练
        epoch_losses = []
        for step in range(steps_per_epoch):
            # 模拟数据
            batch = {
                'images': torch.randn(4, 100),
                'actions': torch.randn(4, 70)
            }
            
            loss = train_step(model, batch, optimizer, criterion)
            epoch_losses.append(loss)
        
        # 记录
        train_loss = np.mean(epoch_losses)
        val_loss = train_loss * (1 + np.random.randn() * 0.1)  # 模拟验证损失
        
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        learning_rates.append(optimizer.param_groups[0]['lr'])
        
        # 更新学习率
        scheduler.step()
        
        if epoch % 2 == 0:
            print(f"Epoch {epoch}: Train Loss={train_loss:.4f}, Val Loss={val_loss:.4f}, LR={learning_rates[-1]:.6f}")
    
    return train_losses, val_losses, learning_rates

# 运行模拟训练
print("开始模拟训练...\n")
train_losses, val_losses, learning_rates = simulate_training(num_epochs=10)
print("\n✅ 训练完成！")

## 6. 训练可视化

In [ ]:
# 可视化训练过程
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 损失曲线
axes[0].plot(train_losses, label='Train Loss', marker='o')
axes[0].plot(val_losses, label='Val Loss', marker='s')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 学习率曲线
axes[1].plot(learning_rates, marker='o', color='green')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Learning Rate')
axes[1].set_title('Learning Rate Schedule')
axes[1].set_yscale('log')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. 训练技巧

In [ ]:
print("""
## 训练技巧总结

### 1. 学习率调度
- Warmup: 前 N 步线性增加学习率
- Cosine Annealing: 余弦退火衰减
- Step Decay: 每隔 N epoch 衰减

### 2. 正则化
- Weight Decay: 权重衰减 (0.01-0.1)
- Dropout: 随机失活 (0.1-0.3)
- Gradient Clipping: 梯度裁剪 (max_norm=1.0)

### 3. 优化技巧
- Mixed Precision: 混合精度训练 (FP16)
- Gradient Accumulation: 梯度累积 (大 batch 模拟)
- EMA: 指数移动平均 (模型参数平滑)

### 4. 数据相关
- Data Augmentation: 数据增强
- Normalization: 输入归一化
- Shuffling: 每 epoch 打乱数据

### 5. 检查点管理
- Save Best: 保存验证损失最低的模型
- Save Regular: 定期保存
- Resume: 支持断点续训
""")

## 8. 完整训练脚本

In [ ]:
print("""
完整训练脚本示例:

```python
# scripts/train.py
import torch
from src.models.vla_model import build_vla_model
from src.training.trainer import Trainer
from src.data.dataset import VLADataset

# 加载配置
config = load_config('configs/model/pi0_base.yaml')

# 创建模型
model = build_vla_model(config['model'])

# 创建数据集
train_dataset = VLADataset(
    data_path='data/droid',
    split='train',
    **config['data']
)

val_dataset = VLADataset(
    data_path='data/droid',
    split='val',
    **config['data']
)

# 创建训练器
trainer = Trainer(
    model=model,
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    config=config['training']
)

# 开始训练
trainer.train()
```
""")

print("🎉 训练教程完成！")
print("\n下一步: notebooks/03_inference.ipynb")